**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Computer Vision — Lezione 17
## Dense prediction
16 novembre 2026. Esperimenti numerici sintetici assegnati, senza addestramento né inferenza di reti. Cxx conta tutte le celle, inclusi titoli e testo.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.


**Lessico iniziale.** Feature: rappresentazione numerica; encoder: estrattore; decoder: uscita spaziale; upsampling: aumento della griglia. Foreground è la regione positiva, background lo sfondo, GT l’annotazione di riferimento. IoU è intersezione/unione, mIoU la media per classe dichiarata. Gli score sono indicati con s nel testo; eventuali nomi di variabili nel codice sono alias della stessa quantità. Warp significa ricampionare una seconda immagine sulla griglia della prima; EPE misura l’errore geometrico del flusso.

<!-- C02 -->
## Avvio in Colab o Jupyter
Caricare questo singolo notebook in Colab e scegliere runtime Python CPU. Non occorrono immagini esterne, pesi, Drive o GPU. Colab include NumPy e Matplotlib. In locale installare le dipendenze indicate nel README. Eseguire C03 e poi i blocchi della percorso. Salvare notebook e JSON prodotto da C27 prima di chiudere il runtime. Le immagini sono mappe numeriche di esempi assegnati, non prestazioni di un modello.

In [ ]:
# C03
import sys,json,time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6,suppress=True)
START=time.perf_counter()
RESULTS={"python":sys.version.split()[0],"numpy":np.__version__,"protocollo":"esperimenti sintetici assegnati, CPU, nessuna rete"}
OUT=Path("output_lezione17");OUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size":13,"figure.dpi":120})
def savefig(name):
    plt.tight_layout();plt.savefig(OUT/(name+".png"),dpi=180,bbox_inches="tight");plt.show()
print("CPU pronta, nessun download",RESULTS)


<!-- C04 -->
## N01 — Classe e identità
C05 richiede C03. Due oggetti della stessa classe hanno ID 1 e 2. Cosa si perde convertendo gli ID in una maschera binaria? Può bastare contare le componenti connesse?

In [ ]:
# C05
instances=np.zeros((8,12),dtype=int)
instances[2:6,2:6]=1;instances[2:6,6:10]=2
semantic=(instances>0).astype(int)
fig,axes=plt.subplots(1,2,figsize=(10,3.7))
for ax,arr,title in zip(axes,[instances,semantic],["ID di istanza: 1 e 2","Classe: background / oggetto"]):
    ax.imshow(arr,cmap="viridis",vmin=0,vmax=2,interpolation="nearest");ax.set_title(title);ax.set_xlabel("x (colonna)");ax.set_ylabel("y (riga)")
savefig("identita")
RESULTS["identita"]={"istanze":2,"pixel_foreground":int(semantic.sum()),"id_semantici":np.unique(semantic).tolist()}


<!-- C06 -->
## N02 — Logit, softmax e cross entropy
C07 richiede C03. Per logit (2, 1, 0), calcolare le probabilità. Quanto vale la CE se la classe vera è 0? E se è 2? Perché non si applica argmax prima della loss?

In [ ]:
# C07
logits=np.array([2.,1.,0.]);e=np.exp(logits-logits.max());prob=e/e.sum()
ce0=-np.log(prob[0]);ce2=-np.log(prob[2])
print("probabilità",prob,"CE y=0",ce0,"CE y=2",ce2)
assert np.allclose(prob.sum(),1) and np.isclose(ce2-ce0,2)
RESULTS["softmax"]={"p":prob.tolist(),"ce0":float(ce0),"ce2":float(ce2)}


<!-- C08 -->
## N03 — Confusione, IoU e Dice
C09 richiede C03. Righe=GT, colonne=predizione. Per matrice [[8, 2], [1, 5]], ricavare TP/FP/FN della classe 1, IoU per classe, mIoU, Dice foreground e pixel accuracy. Quali casi vuoti richiedono una convenzione?

In [ ]:
# C09
def segmentation_metrics(cm):
    cm=np.asarray(cm,dtype=np.int64)
    tp=np.diag(cm);union=cm.sum(0)+cm.sum(1)-tp
    iou=np.divide(tp,union,out=np.full(tp.shape,np.nan,dtype=float),where=union>0)
    den=cm.sum(0)+cm.sum(1)
    dice=np.divide(2*tp,den,out=np.full(tp.shape,np.nan,dtype=float),where=den>0)
    return iou,dice,float(np.trace(cm)/cm.sum()) if cm.sum() else np.nan
cm=np.array([[8,2],[1,5]])
iou,dice,acc=segmentation_metrics(cm)
print("confusione GT righe",cm,"IoU",iou,"mIoU",np.nanmean(iou),"Dice",dice,"accuracy",acc,sep="\n")
assert np.isclose(iou[1],5/8) and np.isclose(dice[1],10/13)
RESULTS["metriche"]={"confusione":cm.tolist(),"iou":iou.tolist(),"miou":float(np.nanmean(iou)),"dice":dice.tolist(),"accuracy":acc}


<!-- C10 -->
## N04 — Classe rara e aggregazione
C11 richiede C03/C09. In 1000 pixel, 20 appartengono al foreground. Valutare la predizione tutta background. Poi confrontare IoU media per immagine e IoU su conteggi aggregati per (TP, FP, FN)=(1, 0, 1) e (90, 10, 0).

In [ ]:
# C11
rare=np.array([[980,0],[20,0]]);ri,rd,ra=segmentation_metrics(rare)
per_image=np.array([1/2,90/100]);global_iou=91/102
print("baseline background: accuracy",ra,"IoU",ri,"mIoU",np.nanmean(ri))
print("IoU media immagini",per_image.mean(),"IoU aggregata",global_iou)
RESULTS["imbalance"]={"accuracy":ra,"iou":ri.tolist(),"miou":float(np.nanmean(ri)),"image_mean":float(per_image.mean()),"global_iou":global_iou}


<!-- C12 -->
## N05 — Dimensioni di una U-Net compatta
C13 richiede C03. Variante didattica con padding same, due downsampling 2×, canali 8/16/32. Calcolare le forme di encoder, upsampling e concatenazioni su input 128×128×3. Non si esegue una rete.

In [ ]:
# C13
shapes={"input":(128,128,3),"encoder1":(128,128,8),"encoder2":(64,64,16),"bottleneck":(32,32,32),"up2":(64,64,16),"concat2":(64,64,32),"decoder2":(64,64,16),"up1":(128,128,8),"concat1":(128,128,16),"decoder1":(128,128,8),"logits":(128,128,2)}
for name,shape in shapes.items():print(name,shape)
assert shapes["concat2"][-1]==shapes["up2"][-1]+shapes["encoder2"][-1]
RESULTS["forme_unet_same"]={k:list(v) for k,v in shapes.items()}


<!-- C14 -->
## N06 — Interpolazione delle etichette
C15 richiede C03. I codici 0 e 2 identificano categorie nominali. Che cosa produce l’interpolazione lineare al punto medio? Perché interpolare prima i logit e poi applicare argmax è diverso?

In [ ]:
# C15
labels=np.array([0.,2.]);x=np.linspace(0,1,9)
linear=np.interp(x,[0,1],labels);nearest=labels[(x>=.5).astype(int)]
print("coordinate",x,"interpolazione lineare degli ID",linear,"nearest",nearest,sep="\n")
fig,ax=plt.subplots(figsize=(10,3.8));ax.plot(x,linear,"o-",label="Lineare su ID nominali");ax.step(x,nearest,where="post",label="Nearest");ax.set(xlabel="Coordinata normalizzata x",ylabel="Codice di classe",yticks=[0,1,2]);ax.legend();savefig("resize_etichette")
RESULTS["resize"]={"midpoint_linear":float(linear[4]),"nearest_ids":np.unique(nearest).tolist()}


<!-- C16 -->
## N07 — Convoluzione trasposta e copertura
C17 richiede C03. Per input[1, 1, 1], kernel unitario di lunghezza 3 e stride 2, sommare i contributi. Cambiare solo il kernel a lunghezza 4. Quali differenze emergono fra interno e bordi?

In [ ]:
# C17
def transposed_ones(length,kernel,stride):
    out=np.zeros((length-1)*stride+kernel,dtype=int)
    for i in range(length):out[i*stride:i*stride+kernel]+=1
    return out
cov3=transposed_ones(3,3,2);cov4=transposed_ones(3,4,2)
print("k=3,s=2",cov3,"k=4,s=2",cov4)
fig,axes=plt.subplots(1,2,figsize=(10,3.8))
for ax,a,k in zip(axes,[cov3,cov4],[3,4]):ax.stem(np.arange(len(a)),a);ax.set(xlabel="Posizione di output",ylabel="Contributi",title=f"Kernel {k}, stride 2",ylim=(0,2.5))
savefig("copertura_trasposta")
RESULTS["trasposta"]={"k3":cov3.tolist(),"k4":cov4.tolist()}


<!-- C18 -->
## N08 — Box e maschera
C19 richiede C03. Una maschera 4×4 piena e una maschera diagonale 4×4 hanno lo stesso bounding box. Calcolare IoU dei box e IoU delle maschere. Il mask AP misura lo stesso fenomeno del bbox AP?

In [ ]:
# C19
full=np.ones((4,4),bool);diag=np.eye(4,dtype=bool)
mask_iou=np.logical_and(full,diag).sum()/np.logical_or(full,diag).sum()
print("IoU box=1; IoU mask=",mask_iou)
fig,axes=plt.subplots(1,2,figsize=(9,3.8))
for ax,a,title in zip(axes,[full,diag],["Maschera piena","Maschera diagonale"]):ax.imshow(a,cmap="gray",vmin=0,vmax=1,interpolation="nearest");ax.set(title=title,xlabel="x (colonna)",ylabel="y (riga)",xticks=range(4),yticks=range(4))
savefig("box_vs_mask")
RESULTS["box_mask"]={"bbox_iou":1.,"mask_iou":float(mask_iou)}


<!-- C20 -->
## N09 — Profondità metrica e scala
C21 richiede C03. GT z=(1, 2, 4)m, predizione(2, 4, 8)m. Calcolare AbsRel, RMSE e varianza degli errori log. Ripetere dopo allineamento della scala tramite rapporto delle mediane.

In [ ]:
# C21
z=np.array([1.,2.,4.]);zhat=2*z
absrel=np.mean(abs(zhat-z)/z);rmse=np.sqrt(np.mean((zhat-z)**2));d=np.log(zhat)-np.log(z)
si=np.mean(d*d)-np.mean(d)**2;scale=np.median(z)/np.median(zhat)
print("AbsRel",absrel,"RMSE metri",rmse,"SI log, lambda=1",si,"scala mediana",scale)
assert np.allclose(zhat*scale,z)
RESULTS["depth"]={"gt":z.tolist(),"pred":zhat.tolist(),"absrel":float(absrel),"rmse_m":float(rmse),"si_log":float(si),"median_scale":float(scale)}


<!-- C22 -->
## N10 — Flusso, warp e validità
C23 richiede C03. Una traslazione assegnata sposta i pixel di u=2 a destra, v=1 in basso. Ricostruire I1 campionando I2 in(x+u, y+v). Contare i pixel validi e calcolare EPE per predizione(1, 1). Come cambiano i vettori dopo resize isotropo 0.5?

In [ ]:
# C23
rng=np.random.default_rng(20261116);I1=rng.random((12,16));I2=np.zeros_like(I1);I2[1:,2:]=I1[:-1,:-2]
valid=np.zeros_like(I1,dtype=bool);valid[:-1,:-2]=True
warped=np.full_like(I1,np.nan);warped[:-1,:-2]=I2[1:,2:]
error=float(np.max(abs(warped[valid]-I1[valid])));epe=float(np.linalg.norm(np.array([1.,1.])-np.array([2.,1.])))
fig,axes=plt.subplots(1,3,figsize=(12,3.3))
for ax,a,title in zip(axes,[I1,I2,warped],["I1 sintetica","I2 traslata (+2,+1)","I2 campionata con il flow"]):ax.imshow(a,cmap="gray",vmin=0,vmax=1);ax.set(title=title,xlabel="x",ylabel="y")
savefig("flow_warp")
print("validi",valid.sum(),"errore max valido",error,"EPE pixel",epe)
RESULTS["flow"]={"uv":[2,1],"valid_pixels":int(valid.sum()),"warp_max_error":error,"epe_px":epe,"half_size_uv":[1.,.5]}


<!-- C24 -->
## N11 — Aperture problem
C25 richiede C03. Considerare gradienti (1, 0), (2, 0), (3, 0) e variazioni temporali−2, −4, −6. Determinare rango, autovalori e soluzione ai minimi quadrati. Aggiungere il gradiente(0, 1) con It=−1.

In [ ]:
# C25
A=np.array([[1.,0.],[2.,0.],[3.,0.]]);b=np.array([2.,4.,6.])
rank=int(np.linalg.matrix_rank(A));eig=np.linalg.eigvalsh(A.T@A);sol=np.linalg.lstsq(A,b,rcond=None)[0]
A2=np.vstack([A,[0.,1.]]);b2=np.r_[b,1.];sol2=np.linalg.lstsq(A2,b2,rcond=None)[0]
print("rango",rank,"autovalori",eig,"soluzione norma minima",sol,"con secondo orientamento",sol2)
RESULTS["aperture"]={"rank":rank,"eigenvalues":eig.tolist(),"minimum_norm":sol.tolist(),"rank2_solution":sol2.tolist()}


<!-- C26 -->
## N12 — Report e limiti
C27 richiede C03 e i blocchi di cui si vogliono salvare i risultati. Verificare i nomi delle sezioni prima di esportare. Quali risultati potrebbero essere attribuiti a una rete addestrata? Salvare il JSON dal pannello File di Colab.

In [ ]:
# C27
RESULTS["elapsed_notebook_seconds"]=time.perf_counter()-START
path=OUT/"risultati_lezione17.json";path.write_text(json.dumps(RESULTS,indent=2,ensure_ascii=False),encoding="utf-8")
print("Salvato:",path.resolve());print("Sezioni:",list(RESULTS))
print("Colab: File → scarica JSON e notebook. Il runtime è temporaneo.")
